# Score-Based IR Image Super-Resolution Training (Colab)

This notebook trains an NCSNv2 score model on IR images using Google Colab's GPU.

**Setup steps:**
1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU or better)
2. Upload your repo to Google Drive OR clone from GitHub
3. Upload your training images to Drive
4. Run all cells

## 1. GPU Check & Setup

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    raise RuntimeError("No GPU found! Go to Runtime -> Change runtime type -> GPU")

## 2. Mount Google Drive

Your training images and checkpoints will be stored on Drive so they persist between sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone / Update Repository

**Option A:** Clone from GitHub (edit the URL if your repo is private, add a token).

**Option B:** If you uploaded the repo folder to Drive, skip this cell and adjust `REPO_DIR` below.

In [ ]:
import os

# === EDIT THESE ===
GITHUB_REPO = "https://github.com/YOUR_USERNAME/Sparse-recovery-of-IR-images-using-a-score-based-model.git"
REPO_DIR = "/content/sparse-ir"  # Where to clone the repo on Colab

if os.path.exists(REPO_DIR):
    print(f"Repo already exists at {REPO_DIR}, pulling latest...")
    !cd {REPO_DIR} && git pull
else:
    !git clone {GITHUB_REPO} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls

In [ ]:
# Install any missing dependencies
!pip install -q pyyaml opencv-python-headless pillow matplotlib

## 4. Upload Training Images

You need to get your training images onto Colab. Choose one option:

**Option A (Recommended):** Zip your `images/` folder locally, upload it to Google Drive, and unzip here.

**Option B:** Use the raw data and run `process_data_set.py` on Colab (slower, requires the raw FLIR data).

In [ ]:
# === Option A: Unzip from Google Drive ===
# First, upload images.zip to your Google Drive root folder
# The zip should contain the images/ folder structure:
#   images/high_res_train/*.jpg
#   images/high_res_test/*.jpg
#   images/medium_res_train/*.jpg
#   images/medium_res_test/*.jpg
#   ... etc

DRIVE_ZIP = "/content/drive/MyDrive/images.zip"  # <-- Edit this path

if os.path.exists(DRIVE_ZIP):
    print(f"Unzipping {DRIVE_ZIP}...")
    !unzip -q -o {DRIVE_ZIP} -d {REPO_DIR}
    print("Done!")
elif os.path.exists(os.path.join(REPO_DIR, 'images')):
    print("images/ folder already exists, skipping.")
else:
    print(f"WARNING: {DRIVE_ZIP} not found and no images/ folder exists!")
    print("Please upload images.zip to your Google Drive or adjust DRIVE_ZIP path.")
    print("")
    print("To create images.zip locally, run in PowerShell:")
    print('  Compress-Archive -Path .\\images\\* -DestinationPath images.zip')

In [ ]:
# Verify images are present
import cv2
for d in ['high_res_train', 'half_res_train', 'medium_res_train', 'low_res_train']:
    p = f'images/{d}'
    if os.path.isdir(p):
        files = os.listdir(p)
        if files:
            img = cv2.imread(os.path.join(p, files[0]), cv2.IMREAD_GRAYSCALE)
            print(f"{d}: {len(files)} images, {img.shape[1]}x{img.shape[0]}")
        else:
            print(f"{d}: empty")
    else:
        print(f"{d}: NOT FOUND")

## 5. Choose Config

Pick which model to train. Higher resolution = more VRAM + longer training.

| Config | Target | Train Data | ~Params | VRAM needed |
|--------|--------|------------|---------|-------------|
| `ir_32_from_16.yml` | 32×32 | medium_res | 20M | ~2 GB |
| `ir_64_from_32.yml` | 64×64 | half_res | 47M | ~6 GB |
| `ir_128_from_32.yml` | 128×128 | high_res | 83M | ~12 GB |
| `ir_256_from_64.yml` | 256×256 | half_res | 83M | ~20 GB |
| `ir_512_from_128.yml` | 512×512 | high_res | 83M | ~40 GB (A100) |

In [ ]:
# === EDIT THIS: Choose your config ===
CONFIG = "configs/ir_128_from_32.yml"

# Print the config
with open(CONFIG) as f:
    print(f.read())

## 6. Train!

Checkpoints are saved to `checkpoints/<config_name>/` and also copied to Google Drive periodically so you don't lose progress if the session disconnects.

In [ ]:
import shutil
from pathlib import Path

# Derive checkpoint dir name from config
config_name = Path(CONFIG).stem  # e.g. 'ir_128_from_32'
LOCAL_CKPT_DIR = f"checkpoints/{config_name}"
DRIVE_CKPT_DIR = f"/content/drive/MyDrive/ir_checkpoints/{config_name}"

# Check if there's a Drive checkpoint to resume from
drive_latest = os.path.join(DRIVE_CKPT_DIR, 'latest.pth')
local_latest = os.path.join(LOCAL_CKPT_DIR, 'latest.pth')

if os.path.exists(drive_latest) and not os.path.exists(local_latest):
    print(f"Found checkpoint on Drive, copying to local...")
    os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)
    shutil.copy2(drive_latest, local_latest)
    print(f"Copied {drive_latest} -> {local_latest}")
elif os.path.exists(local_latest):
    print(f"Local checkpoint exists, will resume from it.")
else:
    print(f"No checkpoint found, training from scratch.")

print(f"\nConfig: {CONFIG}")
print(f"Local checkpoints: {LOCAL_CKPT_DIR}")
print(f"Drive backup: {DRIVE_CKPT_DIR}")

In [ ]:
# Run training
!python ir_ncsn_main.py --task train --config {CONFIG}

In [ ]:
# Backup checkpoint to Google Drive (run this periodically or after training)
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

for fname in os.listdir(LOCAL_CKPT_DIR):
    src = os.path.join(LOCAL_CKPT_DIR, fname)
    dst = os.path.join(DRIVE_CKPT_DIR, fname)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
        print(f"Backed up: {fname}")

print(f"\nCheckpoints backed up to {DRIVE_CKPT_DIR}")

## 7. Reconstruction (Optional)

After training, run reconstruction to test the model. You can change CONFIG to a different measurement size while using the same checkpoint (same target size).

In [ ]:
# For reconstruction with a different measurement size, override checkpoint_dir
# to point to the trained model's checkpoint.
# Example: model was trained with ir_128_from_32, reconstruct with ir_128_from_16

RECON_CONFIG = CONFIG  # or e.g. "configs/ir_128_from_16.yml"
RECON_CKPT = LOCAL_CKPT_DIR  # reuse the trained model checkpoint

!python ir_ncsn_main.py --task reconstruct --config {RECON_CONFIG} --checkpoint_dir {RECON_CKPT}

In [ ]:
# Display reconstruction results
from IPython.display import Image, display
import glob as g

recon_name = Path(RECON_CONFIG).stem
result_dir = f"results/{recon_name}"

for img_path in sorted(g.glob(os.path.join(result_dir, '*.png')))[:5]:
    print(img_path)
    display(Image(filename=img_path, width=600))

## 8. Download Checkpoint Locally

After training, download the checkpoint to your local machine.

In [ ]:
# Option 1: Download directly from Colab
from google.colab import files

ckpt_to_download = os.path.join(LOCAL_CKPT_DIR, 'latest.pth')
if os.path.exists(ckpt_to_download):
    files.download(ckpt_to_download)
    print(f"Downloading {ckpt_to_download}")
else:
    print("No checkpoint found. Train first!")

# Option 2: Already backed up to Drive (see cell above)
print(f"\nAlso available on Drive at: {DRIVE_CKPT_DIR}")